# Demo Clinic：多维度体验 → 图文报告 → 自动修复 → 复测

本 Notebook 提供一套面向 Web demo 的体验与修复工作流。运行前，需要配置待测项目、产品要求和测试任务。

工作流分为五个阶段：

1. 分析 agent 动态规划分层任务，Python 按依赖派发给体验 agents。
2. 记录问题、操作步骤和截图。
3. 重放操作，确认问题能否复现。
4. 修复 agent 根据问题报告修改源码副本。
5. 复测并生成图文报告。

| 角色 | 默认模型 | 可访问的内容 |
|---|---|---|
| 客观严谨的体验检查者 | `glm-5.3-flash` | 目标网页、截图和页面元素 |
| 修复者 | `glm-5.3` | 已复现的文字问题单和源码副本 |

用户上传网站源码 ZIP，选择完整流程或仅体验并出报告。程序解压项目，并在独立副本中启动、测试和修复。

使用方法：在本地 Jupyter 中选择 Python 3，第 3 步设置模型与提示词，第 4 步上传 ZIP 并填写项目信息，点击提交成功后按顺序继续。第 5 步通过隐藏输入框填写 API Key，不要把密钥写进代码。

体验标准来自同文件夹的 `体验Agent检查维度.md`，修改后下次运行体验阶段即生效。首次选择一个核心任务，确认工具连接正常后再扩大范围。第十步由用户按编号选择问题，逐条修复并复测。

交付前，请实际运行工作流，检查结果并保存 Notebook。

## 1 · 解包程序，建立运行目录

本单元格将 Notebook 内打包的通用程序解压到：

`DemoClinic运行目录/implementation`

其中包括：

- `workflow_engine.py`：安排测试、修复、复测和报告生成。
- `task_queue.py`：生成分层任务卡，按依赖分发体验任务。
- `project_submission.py`：显示上传表单，解压与导出项目。
- `experience_skill.py`：按需读取检查规范。
- `browser_worker.py`：操作浏览器、截图和执行检查。
- `browser_capabilities.py`：扩展浏览器工具、会话、环境模拟和覆盖记录。
- `体验Agent检查维度.md`：检查标准和执行规范；同文件夹已存在的版本不会被覆盖。
- `requirements.txt`：记录需要安装的 Python 工具。

本步骤不包含待测项目。目标项目将在后续配置中指定。

运行本单元格只会解包和加载程序，不会调用模型或开始测试。重复运行会覆盖解包目录中的同名程序文件。

In [ ]:
from pathlib import Path
import base64, io, zipfile, sys, os, importlib, json, datetime, shutil
from IPython.display import display, HTML, Markdown

WORK_ROOT = Path.cwd() / "DemoClinic运行目录"
IMPLEMENTATION = WORK_ROOT / "implementation"
IMPLEMENTATION.mkdir(parents=True, exist_ok=True)
_bundle = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_bundle))) as z:
    for info in z.infolist():
        destination = (IMPLEMENTATION / info.filename).resolve()
        if IMPLEMENTATION.resolve() not in destination.parents:
            raise ValueError("附带文件路径异常")
        destination.parent.mkdir(parents=True, exist_ok=True)
        if info.filename in ("体验Agent检查维度.md", "修复Agent执行规范.md"):
            # Preserve the user's editable protocol; bundled copy is only a fallback.
            editable = Path.cwd() / info.filename
            if not editable.exists():
                editable.write_bytes(z.read(info))
            destination.write_bytes(editable.read_bytes())
        else:
            destination.write_bytes(z.read(info))
if str(IMPLEMENTATION) not in sys.path:
    sys.path.insert(0, str(IMPLEMENTATION))
import workflow_engine as wf
importlib.reload(wf)
print("运行目录：", WORK_ROOT)
print("实现代码：", IMPLEMENTATION)


## 2 · 安装并检查浏览器

首次运行需要联网下载 Python 依赖和 Chromium。之后会复用本机浏览器缓存。
浏览器在独立进程里运行，以兼容 Windows 下 Jupyter 的事件循环。

In [ ]:
import subprocess, importlib.util
for module, package in [("playwright", "playwright>=1.55,<2"), ("ipywidgets", "ipywidgets>=8,<9")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", package], check=True)
subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)
print("浏览器和 ZIP 上传表单已就绪。若下一步表单无法显示，请重启 Jupyter 后重跑。")


## 3 · 配置体验 agent 与工作流

**这一部分由工作流的开发者设置，通常保持默认即可。这里决定 agent 怎样工作；下一步再由提交者填写这次要检查的项目。运行本格不会调用模型，也不会启动体验。**

### 体验提示词是什么？

体验提示词存放在同目录的 `体验Agent检查维度.md` 中，相当于给体验 agent 的工作说明书。它规定检查顺序、判断依据、可用工具的使用方法，以及报告需要保留哪些证据。

当前提示词要求 agent 作为一位**客观、严谨的 Web 产品体验检查者**，依据真实浏览器操作作判断。它包含 42 个候选检查维度，例如首次使用、布局、导航、表单、搜索、数据保存、账号、文件、媒体、AI 功能及异常恢复。agent 会根据产品实际功能选择适用项，不要求每个项目都检查全部 42 项。

### 它按什么顺序检查？

| 阶段 | 主要工作 |
|---|---|
| 产品认识 | 先看首页，记录自己对用途与入口的理解，再读取用户提交的项目要求 |
| 核心流程 | 完成用户指定的核心任务，并检查最终结果 |
| 基础交互 | 检查页面布局、输入、按钮、导航等 |
| 专项功能 | 按实际功能检查账号、文件、图表、媒体、AI 等适用内容 |
| 边界与恢复 | 在正常流程基础上测试异常输入；允许时模拟故障并恢复 |
| 复核与报告 | 整理问题、截图、检查点与覆盖记录，说明哪些没有测试 |

### 提示词怎样约束判断？

- 以用户提供的成功标准和页面明确承诺为依据，不自行发明产品要求。
- 先实际操作，再判断结果；看到“成功”提示，还应核对实际数据或产物。
- 区分功能错误、疑似问题与体验建议，建议不会自动触发改代码。
- 每个问题记录实际输入、操作步骤、预期、结果和证据；没有测过就写“未测试”。
- 提示词提到一种工具，不代表程序一定具备它。实际能力以本轮注册的工具为准；缺少工具、账号或预算时要说明限制。

### 下方配置需要怎么看？

| 设置 | 含义 | 第一次运行建议 |
|---|---|---|
| `RUN_MODE` | `live` 调用真实模型，`preview` 只检查网页预览 | 正式体验使用 `live` |
| `HEADLESS` | 是否隐藏浏览器窗口 | `False`，便于看见操作 |
| `MAX_TEST_TURNS` / `MAX_FIX_TURNS` | 体验总轮数与每张修复卡片的轮数上限 | 先保持默认；轮数不等于点击次数 |
| `test_model` / `fix_model` | 分别选择体验、修复模型 | 按已开通的接口设置 |
| `vision` | 是否把截图发送给体验模型 | 只有支持图片输入的模型才能开启 |
| `prompt_path` | 指向你的体验提示词 Markdown 文件 | 保持路径指向实际文件 |
| `browser_engine` | 使用哪一种浏览器 | 默认 Chromium |
| `max_tool_calls` / `max_test_seconds` | 工具次数和体验阶段时间预算 | 先保持默认，预算不保证覆盖全部维度 |
| `EXTRA_ALLOWED_ORIGINS` | demo 必需的其他接口域名 | 需要外部接口时再配置 |
| `allow_fault_injection` / `allow_permissions` | 是否允许故障、定位和权限模拟 | 默认关闭，测试环境明确允许后再开 |

**本格完成后只准备好运行设置。提示词正文会在后面的体验阶段读取，并保存本轮使用的快照。**调整提示词时修改 Markdown 文件；调整模型或预算时修改本格。项目介绍和任务请放在第 4 步。


### 按需读取手册：API 是怎样调用 skill 的？

本工作流把 `体验Agent检查维度.md` 当作本地 skill 手册，不需要平台专门支持“skill API”。Python 提供 `read_skill` 工具：模型请求某个维度编号，程序读取对应章节，并把内容作为工具结果发回模型。

- 初始提示词只有通用规则、42项标题索引、阶段顺序和运行预算。
- 体验时按需读取当前1—3个维度；专项工具也按需启用。
- 发现问题时读取问题卡片规范，调用 `record_issue` 即时保存卡片。
- `read_memory` 可以读取已保存的问题、检查点和进度；不会把全部历史一直塞进对话。
- 体验结束后，另一次只读整理会话使用同一个体验模型，读取报告章节和已保存卡片，提交结构化报告内容。这里有额外的 `max_report_turns` 调用预算。

本轮 `baseline_comprehensive` 文件夹中的 `issue_cards.json` 保存问题卡片，`checkpoints.json` 保存检查点，`run_state.json` 保存进度，`skill_reads.json` 记录读取过的章节，`report_content.json` 保存整理结果。这些记录都由程序创建，无需你手工管理。

目前支持保存和重读记录，不支持重启后自动恢复原来的浏览器现场。JSON结构与引用可以校验，但模型文字结论仍需依据证据审阅。

### 本版本的模型与接口

- 体验与报告整理：GLM `glm-5.3-flash`。
- 源码修复：GLM `glm-5.3`。
- 下方代码直接指定 `test_model`（体验/报告）和 `fix_model`（修复）。API Key 只验证权限，不决定型号。
- 两端默认使用智谱官方标准接口 `https://open.bigmodel.cn/api/paas/v4`。第三方平台的 Key 必须配套其接口地址和准确模型 ID，不能直接用于官方接口。
- 两个角色分别读取自己的 Key；同一账号的 Key 如果有两个型号的权限，可以在第五步输入同一个 Key。
- `vision=True` 会向体验模型发送截图。第六步只验证文字连接，不代表截图与工具调用兼容性已验证。
- 修改此格后重新运行第三步，再到第四步重新提交，避免本次任务保留旧配置。

### 用量与上下文

默认只发送一份当前页面信息，历史保留最近两组工具交互摘要。输入普通表单字段时不重复发送截图；页面切换、点击和异常时保留视觉观察。同一 form 内最多8个输入字段可批量填写。剩余轮数会提供给 agent，提醒优先覆盖核心任务并预留收尾。模型仍可能未完成全部任务，报告会如实说明。

每次模型调用的用量、耗时、文字长度和图片数量保存在任务目录的 model_usage.jsonl；不保存请求正文或密钥。图片数量不等于图片 token，接口返回的实际总用量才是计费参考。

### 分层任务调度

第八步由分析 agent 生成通用任务图；体验 agent 的规则按任务选取，不按42项平均分配。分析和体验默认共用 test_model 与体验 Key，但使用独立上下文。MAX_TEST_TURNS 是所有体验任务共享的轮数上限，不是每个 agent 各获得一份。


In [ ]:
# 工作流运行设置：平时可以保持默认，不在这里填写项目资料。
RUN_MODE = "live"          # live：调用模型；preview：只预览网页
HEADLESS = False          # False：显示浏览器窗口
MAX_TEST_TURNS = 60
MAX_FIX_TURNS = 14        # 每张所选问题卡片的模型调用上限
MAX_REPAIR_ISSUES = 1     # 旧版自动选卡上限；新版按用户选择逐条处理
EXTRA_ALLOWED_ORIGINS = []  # 项目必需的外部接口域名，按需配置

RUNTIME_CONFIG = {
    "layered_tasks": True,
    "experience_workers": 2,  # 最多同时工作的体验agent；网址/固定端口自动串行
    "max_analysis_tasks": 12,  # 分析agent最多生成12张任务卡
    "task_max_turns": 15,  # 每张卡片最多分配15轮，所有任务合计仍受MAX_TEST_TURNS限制
    'test_provider': "glm",
    'test_base_url': "https://open.bigmodel.cn/api/paas/v4",
    'test_model': "glm-5.3-flash",  # 体验和报告模型：在这里修改型号
    'fix_provider': "glm",
    'fix_base_url': "https://open.bigmodel.cn/api/paas/v4",
    'fix_model': "glm-5.3",  # 修复模型：在这里修改型号
    'vision': True,
    'extra_allowed_origins': EXTRA_ALLOWED_ORIGINS,
    'headless': HEADLESS,
    'max_test_turns': MAX_TEST_TURNS,
    'max_fix_turns': MAX_FIX_TURNS,
    'max_repairs': MAX_REPAIR_ISSUES,
    'repair_prompt_path': str(Path.cwd() / "修复Agent执行规范.md"),
    'api_timeout': 120,
    'prompt_path': str(Path.cwd() / "体验Agent检查维度.md"),
    'browser_engine': "chromium",
    'max_tool_calls': 180,
    'history_batches': 2,          # 只携带最近2组工具交互摘要，较早证据通过read_memory读取
    'generate_report_content': False, # 默认由 Python 按证据生成简洁报告，不额外调用模型
    'max_report_turns': 10,         # 报告整理的独立调用预算
    'report_max_output_tokens': 6500,
    'max_test_seconds': 1800,
    'allow_fault_injection': False,
    'allow_permissions': False,
}
if RUN_MODE not in ("live", "preview"):
    raise ValueError("RUN_MODE 请填写 live 或 preview")
print("运行设置就绪。下一块填写本次提交的 demo 信息。")

for role, label in (("test", "体验/报告"), ("fix", "修复")):
    print(label, RUNTIME_CONFIG[role + "_model"], RUNTIME_CONFIG[role + "_base_url"])


## 4 · 上传源码 ZIP，填写项目信息

**本步不需要用户填写本机源码路径。**点击运行后会显示项目提交表单，点击“选择项目 ZIP”打开文件选择窗口，选中整个 demo 的源码压缩包。

### 提交流程

1. 将网站项目文件夹压缩为 ZIP，可带一层外部文件夹（如 GitHub 下载的项目包）。不要只打包一个入口而遗漏图片、样式或脚本。
2. 在下方表单选择 ZIP，填写项目信息。
3. 选择完整流程或仅出报告，确认项目允许在本机运行，点击“提交 ZIP 并准备项目”。
4. 看到“提交成功”后继续第 5 步。**表单出现不等于已提交，请逐格运行，不要直接运行全部。**

### 用户填写的维度

| 维度 | 填写内容与示例 |
|---|---|
| 项目名称、用途 | 例如“小组任务看板，帮助成员分配和跟踪课程任务” |
| 目标用户 | 例如“首次使用的大学生小组” |
| 核心任务与成功标准 | 至少一组、最多五组。例如“创建任务并刷新”；成功标准为“标题和日期均保留” |
| 已知问题与未完成功能 | 例如“通知暂未接入”，避免把已知情况当新发现 |
| 设计与操作限制 | 例如“保留现有布局，仅操作测试数据” |
| 重点测试范围 | 例如“桌面创建流程，手机宽度下的表单”；不意味着全部内容必然测完 |
| 账号与数据说明 | 说明测试账号角色、数据准备情况；不要填写密码；说明不会自动完成登录 |
| 额外验收要求 | 例如“截止日期不得早于开始日期”；不是自动执行的测试命令 |
| 服务方式 | 完整体验、修复与复测；或仅体验并出报告 |

### 网站怎样自动启动？

- **纯 HTML/CSS/JS**：识别 `index.html` 或唯一的 HTML 入口，用 Python 自带的网页服务启动，无需项目依赖。
- **简单 Python/Node 服务**：识别常见唯一入口及读取 `PORT` 的启动方式。Node 项目需本机已有 Node。
- **无法识别、固定端口或有依赖**：表单给出原因；展开“高级启动设置”，填写命令列表，例如 `["python", "app.py"]`。如果代码写死了 3000 端口，填固定端口 `3000`，并先关闭原来的服务。命令支持 `{python}`、`{port}` 占位符。
- 不会直接双击 ZIP 内的 `.cmd` 或猜着运行脚本。复杂数据库和远端服务还需要测试环境配置，本版并非任意项目的一键部署器。

### 文件会放在哪里？

每次提交建立新的 `DemoClinic运行目录/outputs/任务编号/`：`uploaded.zip` 保留上传包；`original` 存放解压后的原始项目；体验和修复使用另外的副本。重新提交不会覆盖上次任务。

ZIP 上限 50 MB，解压后上限 200 MB。依赖目录、凭证文件等会跳过并记录；不要上传真实私人数据。该版本在运行 Notebook 的电脑上执行项目，不是公开网站的服务器隔离环境。

用户资料会进入报告。API Key 在第 5 步输入。首次观察仍先看页面，再读取产品说明。


In [ ]:
# 运行本格显示表单。填写后点击“提交 ZIP 并准备项目”。
# 看到“提交成功”以后，再运行第 5 步；不要使用“运行全部”跳过表单提交。
import project_submission as intake
importlib.reload(intake)
SUBMISSION_FORM = intake.show_submission(RUNTIME_CONFIG, WORK_ROOT / "outputs", globals())


## 5 · 输入体验与修复两个 GLM agent 的 API Key

型号已在第三步代码中指定；本步只输入对应平台的密钥。同一账号的 Key 有两个型号权限时，可以输入同一个 Key。仅出报告时无需修复端 Key。

默认接口为智谱官方标准 API。第三方平台 Key 需要先在第三步修改对应接口地址和型号。

密钥隐藏输入，仅保留在运行环境中，不写入项目资料或 Notebook 代码。每次运行会清除上次的两个 Key。


In [ ]:
if "CONFIG" not in globals() or not CONFIG.get("submission_run"):
    raise RuntimeError("请先完成第 4 步上传，并点击提交直到显示提交成功。")
# 检查本次提交是否仍保留修改前的模型配置。
_model_fields = [role + "_" + field for role in ("test", "fix")
                 for field in ("provider", "base_url", "model")] + ["vision"]
if any(CONFIG.get(field) != RUNTIME_CONFIG.get(field) for field in _model_fields):
    raise RuntimeError("模型配置已改变：请重新完成第四步提交，再继续，避免调用旧模型。")
from getpass import getpass
os.environ.pop("DEMO_TEST_KEY", None)
os.environ.pop("DEMO_FIX_KEY", None)
if RUN_MODE == "live":
    try:
        os.environ["DEMO_TEST_KEY"] = getpass("体验 agent 的 GLM API Key：").strip()
        if CONFIG.get("repair_enabled"):
            os.environ["DEMO_FIX_KEY"] = getpass("修复 agent 的 GLM（智谱）API Key：").strip()
        if not os.environ.get("DEMO_TEST_KEY") or (CONFIG.get("repair_enabled") and not os.environ.get("DEMO_FIX_KEY")):
            raise ValueError("密钥不能为空")
    except BaseException:
        os.environ.pop("DEMO_TEST_KEY", None)
        os.environ.pop("DEMO_FIX_KEY", None)
        raise
    print("GLM 体验密钥已就绪；" + ("GLM 修复密钥已就绪。" if CONFIG.get("repair_enabled") else "本轮仅出报告，无需修复端 Key。"))
else:
    print("预览模式：不需要密钥，也不调用模型。")


## 6 · 检查接口可用性

各做一次最小文字请求，先发现 Key、余额、模型权限或网络问题，避免运行到中途才失败。
这一步验证文字连接；截图输入与工具调用会在真实体验阶段使用。

In [ ]:
if "CONFIG" not in globals() or not CONFIG.get("submission_run"):
    raise RuntimeError("请先完成第 4 步上传，并点击提交直到显示提交成功。")
# 检查本次提交是否仍保留修改前的模型配置。
_model_fields = [role + "_" + field for role in ("test", "fix")
                 for field in ("provider", "base_url", "model")] + ["vision"]
if any(CONFIG.get(field) != RUNTIME_CONFIG.get(field) for field in _model_fields):
    raise RuntimeError("模型配置已改变：请重新完成第四步提交，再继续，避免调用旧模型。")
if RUN_MODE == "live":
    for role in (("test", "fix") if CONFIG.get("repair_enabled") else ("test",)):
        client = wf.ModelClient(CONFIG, role)
        print("准备检查：", "体验/报告" if role == "test" else "修复", client.model, client.base_url)
        answer = client.request([{"role":"user", "content":"请只回复：连接成功。"}])
        print("体验接口" if role == "test" else "修复接口", client.model, client.base_url, "已响应；用量：", client.usage)
else:
    print("已跳过真实模型连接检查。")

## 7 · 自动启动项目并预览

启动用户提交的源码副本，确认网页能打开，并显示截图。此步不调用模型。启动失败时查看本轮目录的 server.log，调整第四步启动设置后重新提交。


In [ ]:
if "CONFIG" not in globals() or not CONFIG.get("submission_run"):
    raise RuntimeError("请先完成第 4 步上传，并点击提交直到显示提交成功。")
RUN = Path(CONFIG["submission_run"])
# 每次重新预览使用新目录，不覆盖旧截图。
PREVIEW_FOLDER = RUN / ("preview_" + datetime.datetime.now().strftime("%H%M%S_%f"))
CONFIG = dict(CONFIG)
wf.dump(RUN / "config.json", CONFIG)

PREVIEW = wf.target_config(CONFIG, CONFIG.get("source_path"), PREVIEW_FOLDER)
with wf.RunningTarget(PREVIEW) as url:
    preview = wf.worker({"operation":"replay", "url":url,
        "actions":[{"action":"goto","url":url}], "checks":[], "output":PREVIEW["folder"]},
        PREVIEW["folder"], CONFIG, credentials=False)
from IPython.display import Image
display(Image(filename=preview["screenshot"]))
print("本轮输出：", RUN)

## 8 · 分析项目，按依赖分发体验任务

运行本格会调用真实模型。先由分析 agent 读取首页、用户要求、检查维度目录和有限的源码样本，动态生成分层任务卡；不包含任何指定 demo 的功能、页面或选择器。

每张卡片包含目标、成功标准、准备步骤、适用维度及前置任务。Python 校验编号、依赖循环和用户任务覆盖，再计算层级；前置任务通过才允许下游执行。前置失败、受阻、未完成或中断，下游都会标记受阻，不执行、不另报重复缺陷；不依赖它的任务可以继续。

体验 agent 一次只负责一张卡片，只读取相关规则和前置页面记忆。各自使用源码副本和全新浏览器；通过依赖不意味着继承了账号或数据，必须按准备说明建立状态。普通源码项目最多并发2个，网址模式和固定端口自动串行。页面记忆只是线索，必须核对当前页面。

体验总预算仍为 MAX_TEST_TURNS，每张卡片最多 task_max_turns；并发任务提前预留轮数，未用完的预留暂不返还。分析阶段最多额外调用2次。额度不足的任务标为未执行，不会假装通过。预算和并发参数都在第三步配置。

输出 task_plan.json（任务卡）、task_state.json（依赖状态）、baseline.json（汇总）。每轮 experience_batch 目录保留各任务结果及页面记忆。后续第九步照常复现问题，第十步选择修复。修好前置问题不会自动宣称下游通过；需要再次执行相关测试。本版本暂不自动续跑旧队列。


In [ ]:
importlib.reload(wf)
BASELINE = wf.run_baseline(CONFIG, RUN) if RUN_MODE == "live" else []
if RUN_MODE == "preview": print("预览模式：没有生成虚构的体验结果。")
for result in BASELINE:
    print(result["persona"]["name"], "：", result["stop_reason"], "；检查数：", len(result["checks"]))

## 9 · 独立重放问题，生成第一版图文报告

问题发现后，程序在全新副本上重放操作。只有原步骤可执行且正确预期仍不满足，才标为“已复现（按所填预期）”。
主观建议、工具定位失败和无法重现的问题不会直接触发自动修复。预期本身是否合理仍取决于你填写的产品约定。
默认由 Python 根据结构化问题与原始证据直接排版，不额外调用模型写 HTML，生成 `体验与修复报告.html` 和 `修复问题单.json`。复现状态以程序结果为准，疑似问题和体验建议不会自动进入修复队列。


In [ ]:
importlib.reload(wf)
ISSUES = wf.reproduce_issues(CONFIG, RUN, BASELINE) if RUN_MODE == "live" else []
REPORT = wf.render_report(RUN, CONFIG, BASELINE, ISSUES)
for issue in ISSUES: print(issue["id"], issue["title"], "→", issue["status"])
display(Markdown(f"第一版图文报告已生成：`{REPORT}`"))

## 10 · 按编号选择问题，逐条修复并复测

运行下方代码后，先查看问题编号、标题及可修复状态，然后在输入框填写编号，例如 `1,2` 或 `ISSUE-001,ISSUE-002`，顺序就是处理顺序。留空并按 Enter 跳过修复。选择后会调用真实修复 API。

只有已复现的功能错误可以选择。疑似问题需要补充验证，体验建议不自动修改。编号不存在、重复或不可修复时会提示错误，尚未调用模型；修正后可重新运行。

每次提交选择都会创建独立批次，从原始源码副本开始，不覆盖之前的批次。因此本批次想保留的修复都应选中；重复提交可能重复产生模型费用。

处理顺序：修复一张卡片 → 语法检查 → 浏览器复测本问题、此前本批次已修好的问题及原有正常流程 → 通过才保留修改 → 下一张。不通过或中断的当前修改不并入后续源码，候选和证据仍保留供检查。原始项目不变。

`MAX_FIX_TURNS` 是每张卡片的模型调用上限。选卡流程以你明确选择的编号为准，不再受旧的默认一张卡片限制。第十一步展示本步已经完成的验证结果，不重复调用模型或重放浏览器。


In [ ]:
importlib.reload(wf)
# 读取磁盘上最新卡片，包含已完成的补充验证。
ISSUES = wf.load(RUN / "issues.json")
if RUN_MODE == "live" and CONFIG.get("repair_enabled", False):
    for item in ISSUES:
        state = "可修复" if wf.repairable(CONFIG, item) else "暂不可修复：" + item.get("status", "未确认")
        print(item["id"], item["title"], "—", state)
    selected_text = input("输入本批次要修复的问题编号（如 1,2；留空跳过）：")
    SELECTED_ISSUE_IDS = wf.select_repair_issues(selected_text, CONFIG, ISSUES)
    REPAIR = wf.repair_selected(CONFIG, RUN, BASELINE, ISSUES, SELECTED_ISSUE_IDS)
else:
    REPAIR = {"status": "skipped", "reason": "仅出报告或预览模式；未调用修复模型"}
print(REPAIR.get("summary", REPAIR.get("reason", "")))
for item in REPAIR.get("issue_results", []):
    print(item["id"], item["status"], item["summary"])
print("修改的文件：", ", ".join(change["path"] for change in REPAIR.get("changes", [])) or "无")
print("完整差异和逐条证据已保存在本轮任务目录。")


## 11 · 查看逐条复测结果与最终报告

新版第十步已逐条执行修复、问题复测和正常流程回归；本步直接读取该批次结果，不重复复测。

- `verified`：所选问题全部通过。
- `partial_verified`：只有部分问题通过；交付副本只包含通过的修改，未通过项仍需处理。
- `not_verified`：本批次没有问题通过。
- `skipped`：没有选择修复。

如果仍在使用旧版第十步生成的候选，本步会兼容执行一次复测。每张卡片的执行错误和检查结果保存在批次目录内，原始项目不变。


In [ ]:
importlib.reload(wf)  # 载入最新复测逻辑，沿用当前任务和候选代码
VERIFICATION = REPAIR.get("verification")
if VERIFICATION is None:
    VERIFICATION = wf.verify(CONFIG, RUN, BASELINE, ISSUES, REPAIR)
REPORT = wf.render_report(RUN, CONFIG, BASELINE, ISSUES, REPAIR, VERIFICATION)
display(HTML("<h3>最终状态：" + VERIFICATION["status"] + "</h3>"))
print(VERIFICATION.get("reason", ""))
import html
display(HTML('<iframe style="width:100%;height:800px;border:1px solid #ddd;border-radius:12px" srcdoc="'
             + html.escape(REPORT.read_text(encoding="utf-8"), quote=True) + '"></iframe>'))

## 12 · 获取报告与完整项目 ZIP

运行本格，将结果集中整理到本次任务的 **`交付结果`** 文件夹，并在 Notebook 显示可打开或下载的链接。

- **体验与修复报告.html**：给用户阅读，图片已内嵌，可以离线打开。
- **修复问题单.json**：结构化的问题与证据记录。
- **项目 ZIP**：保留源项目的图片、字体、样式和其他资源，覆盖采用的代码修改。通过复测叫“修复后的项目”；有修改但未通过叫“待验证的修改候选”；没有修改则叫“原始项目_未修改”。
- **修改与运行说明.txt**：记录修改文件、启动方式和复测结果。
- 其他复测记录与代码差异按本轮实际结果提供。

原始上传包不会被覆盖。项目 ZIP 不包含被跳过的依赖和凭证目录；使用者仍需按说明准备必要环境。报告中的通过仅针对本次选择的修复与检查，不是整个网站的质量保证。


In [ ]:
# 生成面向用户的报告与项目压缩包。
RESULTS = intake.export_project(RUN, CONFIG, REPAIR, VERIFICATION)
print("本轮任务目录：", RUN)
print("报告和项目 ZIP：", RESULTS)
from IPython.display import FileLink
for result_file in sorted(RESULTS.iterdir()):
    try:
        relative = result_file.resolve().relative_to(Path.cwd().resolve())
        display(FileLink(str(relative).replace("\\", "/")))
    except ValueError:
        print(result_file)
if VERIFICATION.get("status") != "verified":
    print("本轮未达到修复验证通过状态。请查看报告，勿把候选修改当成已修复。")


## 13 · 结束运行，清除本次密钥

所有临时浏览器与 demo 服务都在每个阶段结束时关闭。此处再清除 Notebook 内存中的密钥。
要重跑真实模型，需要回到第 5 步重新输入。若保留输出给组委会，请检查截图和源码中没有私人信息。

In [ ]:
os.environ.pop("DEMO_TEST_KEY", None)
os.environ.pop("DEMO_FIX_KEY", None)
print("内存中的两个密钥已清除。")

## 当前本地版的完整流程

用户上传源码 ZIP 并填写要求 → 解压与识别启动方式 → 自动启动副本 → 体验 → 独立复现 → 修复候选源码 → 复测 → HTML 报告、JSON 问题单与项目 ZIP。

所有项目文件和结果保存在运行 Notebook 的电脑上；模型调用使用配置的云端 API。普通静态网站不需要填写路径或安装项目依赖；不明确的启动方式由高级设置补充。修复只发生在候选副本，原始项目保留。

这是本地原型，不具备面向不可信上传的服务器执行隔离。模型行为和修复效果需要真实运行验证；没有通过的结果会按未完成或待验证展示。
